In [ ]:
import re
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize, RegexpTokenizer
from nltk.corpus import stopwords, wordnet
from nltk.stem import PorterStemmer, LancasterStemmer, RegexpStemmer, WordNetLemmatizer

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')
nltk.download('omw-1.4')

roll_no = 1024170239


In [ ]:
tickets = generate_dataset(roll_no)
processed = [process_ticket(t) for t in tickets]


In [ ]:
rows = []
for i, p in enumerate(processed):
    rows.append({
        "ticket": i+1,
        "number of sentences": len(p["sentences"]),
        "number of tokens": len(p["tokens"]),
        "number of punctuation tokens": len(p["punctuation_tokens"]),
        "number of stopwords": len(p["stopwords"]),
        "number of unique tokens": len(set(p["tokens"])),
        "number of final tokens": len(p["final_tokens"])
    })

df = pd.DataFrame(rows)
total = pd.DataFrame([{
    "ticket": "Total",
    "number of sentences": df["number of sentences"].sum(),
    "number of tokens": df["number of tokens"].sum(),
    "number of punctuation tokens": df["number of punctuation tokens"].sum(),
    "number of stopwords": df["number of stopwords"].sum(),
    "number of unique tokens": df["number of unique tokens"].sum(),
    "number of final tokens": df["number of final tokens"].sum()
}])
pd.concat([df,total],ignore_index=True)


In [ ]:
p = processed[0]
{
    "tokens": p["tokens"],
    "punctuation tokens": p["punctuation_tokens"],
    "stopwords": p["stopwords"],
    "final tokens": p["final_tokens"]
}


In [ ]:
raw = " ".join(tickets)

long_words = re.findall(r'\b[A-Za-z]{6,}\b', raw)
numbers = re.findall(r'\b\d+(?:\.\d+)*\b', raw)
capitalized = re.findall(r'\b[A-Z][a-z]*\b', raw)
vowel_words = re.findall(r'\b[aeiouAEIOU][A-Za-z]*\b', raw)

long_words, numbers, capitalized, vowel_words


In [ ]:
replaced = re.sub(r'\b[\w.-]+@[\w.-]+\.\w+\b', '<EMAIL>', raw)
replaced = re.sub(r'https?://\S+|www\.\S+', '<URL>', replaced)
replaced = re.sub(r'\b(?:\+?\d{1,3}[- ]?)?(?:\d{10}|\d{3}[- ]\d{3}[- ]\d{4})\b', '<PHONE>', replaced)

replaced


In [ ]:
def my_tokenize(text):
    pattern = r'https?://\S+|[\w.-]+@[\w.-]+\.\w+|\+?\d{1,3}[- ]?\d{10}|\d+(?:\.\d+)+|[A-Za-z]+(?:-[A-Za-z]+)+|[A-Za-z]+(?:\'[A-Za-z]+)?|\w+'
    tokenizer = RegexpTokenizer(pattern)
    return tokenizer.tokenize(text)

custom_test = [my_tokenize(t) for t in TEST_LINES]
word_test = [word_tokenize(t) for t in TEST_LINES]

custom_tickets = [my_tokenize(t) for t in tickets]

custom_test, word_test


In [ ]:
differences = []
for i in range(len(TEST_LINES)):
    if custom_test[i] != word_test[i]:
        differences.append({
            "line": TEST_LINES[i],
            "my_tokenize": custom_test[i],
            "word_tokenize": word_test[i]
        })

differences[:3]


In [ ]:
unique_tokens = set()
for p in processed:
    unique_tokens.update(p["final_tokens"])

porter = PorterStemmer()
lancaster = LancasterStemmer()
regexp = RegexpStemmer("ing$", min=4)
lemmatizer = WordNetLemmatizer()

def get_pos(word):
    tag = nltk.pos_tag([word])[0][1]
    if tag.startswith("J"):
        return wordnet.ADJ
    if tag.startswith("V"):
        return wordnet.VERB
    if tag.startswith("N"):
        return wordnet.NOUN
    if tag.startswith("R"):
        return wordnet.ADV
    return wordnet.NOUN

result = []
for w in sorted(unique_tokens):
    row = {
        "word": w,
        "Porter": porter.stem(w),
        "Lancaster": lancaster.stem(w),
        "Regexp": regexp.stem(w),
        "WordNet": lemmatizer.lemmatize(w, get_pos(w))
    }
    if len(set(row.values())) > 1:
        result.append(row)

pd.DataFrame(result)


In [ ]:
fingerprint = ''.join([chr(65+(ord(c)%26)) for c in str(roll_no)])[:8]
fingerprint
